# Baseline: kNN + Logistic Regression

Uses the shared `artifacts/samples.parquet` split.

**Features (per `window_id`):** mean/std of `x`,`y`,`z` + `device` + `sensor`  
**Target:** `activity`  
**Models:** `KNeighborsClassifier`, `LogisticRegression`  
**Metrics (test):** accuracy, F1 macro / micro / weighted, macro ROC-AUC (OvR), classification report

In [12]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path(".").resolve()
ART = ROOT / "artifacts"
SAMPLES_PATH = ART / "samples.parquet"
META_PATH = ART / "split_meta.json"

RANDOM_SEED = 42
N_NEIGHBORS = 5

ACTIVITY_ORDER = list(
    json.loads((ROOT / "activity_map.json").read_text())["ACTIVITY_MAP"].keys()
)

print(f"ROOT={ROOT}")
print(f"samples: {SAMPLES_PATH} exists={SAMPLES_PATH.exists()}")

ROOT=/Users/tomhait/git/tom.aharon-hait/projects/phone_watch_project
samples: /Users/tomhait/git/tom.aharon-hait/projects/phone_watch_project/artifacts/samples.parquet exists=True


## 1. Load shared parquet

In [6]:
samples = pd.read_parquet(SAMPLES_PATH)
if META_PATH.exists():
    meta = json.loads(META_PATH.read_text())
    print("Loaded split_meta.json")
    print("test_user_ids:", meta.get("test_user_ids"))
else:
    meta = {}

train_samples = samples[samples["split"].astype(str) == "train"].copy()
test_samples = samples[samples["split"].astype(str) == "test"].copy()
print(f"train sample rows: {len(train_samples):,} | test sample rows: {len(test_samples):,}")

Loaded split_meta.json
test_user_ids: [1606, 1607, 1618, 1626, 1629, 1639, 1640, 1641, 1643, 1646]
train sample rows: 11,921,624 | test sample rows: 3,063,648


## 2. Aggregate per `window_id`

Features: `mean/std` of X/Y/Z + `device` + `sensor`. Target: `activity`.

In [13]:
def aggregate_windows(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby("window_id", observed=True)
    out = g.agg(
        user_id=("user_id", "first"),
        activity=("activity", "first"),
        device=("device", "first"),
        sensor=("sensor", "first"),
        mean_x=("x", "mean"),
        mean_y=("y", "mean"),
        mean_z=("z", "mean"),
        std_x=("x", "std"),
        std_y=("y", "std"),
        std_z=("z", "std"),
        n_samples=("window_idx", "size"),
    ).reset_index()

    # pandas std uses ddof=1; single-sample windows would be NaN — shouldn't happen with MIN_SAMPLES=50
    out[["std_x", "std_y", "std_z"]] = out[["std_x", "std_y", "std_z"]].fillna(0.0)

    out["activity"] = out["activity"].astype(str)
    out["device"] = out["device"].astype(str)
    out["sensor"] = out["sensor"].astype(str)
    return out


train_win = aggregate_windows(train_samples)
test_win = aggregate_windows(test_samples)

FEATURE_NUM = ["mean_x", "mean_y", "mean_z", "std_x", "std_y", "std_z"]
FEATURE_CAT = ["device", "sensor"]
FEATURE_COLS = FEATURE_NUM + FEATURE_CAT
TARGET = "activity"

print(f"train windows: {len(train_win):,} | test windows: {len(test_win):,}")
display(train_win.head())
print("Activity counts (train):")
display(train_win["activity"].value_counts().reindex(ACTIVITY_ORDER).rename("n").to_frame())

train windows: 102,729 | test windows: 24,772


,window_id,user_id,activity,device,sensor,mean_x,mean_y,mean_z,std_x,std_y,std_z,n_samples
0,1600_phone_accel_A_0000,1600,A,phone,accel,-1.000110,9.619472,0.340593,1.892580,3.986348,2.082760,100
1,1600_phone_accel_A_0001,1600,A,phone,accel,-1.370578,9.543877,0.483633,2.094453,4.081441,2.453686,99
2,1600_phone_accel_A_0002,1600,A,phone,accel,-1.521375,9.593882,0.565173,2.468418,4.123296,2.444694,99
3,1600_phone_accel_A_0003,1600,A,phone,accel,-1.501262,9.581829,0.296152,2.113209,4.048018,2.032577,100
4,1600_phone_accel_A_0004,1600,A,phone,accel,-1.569346,9.383181,0.539201,2.238218,3.856700,2.298824,99


Activity counts (train):


,n
activity,
A,5750
B,5539
C,5575
D,5753
E,5753
F,5537
G,5752
H,5715
I,5750


## 3. Preprocess + models

Numeric features are scaled; `device`/`sensor` are one-hot encoded. Fit transforms on **train only**.

In [14]:
X_train = train_win[FEATURE_COLS]
y_train = train_win[TARGET]
X_test = test_win[FEATURE_COLS]
y_test = test_win[TARGET]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), FEATURE_NUM),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            FEATURE_CAT,
        ),
    ]
)

knn = Pipeline(
    steps=[
        ("prep", preprocess),
        (
            "model",
            KNeighborsClassifier(n_neighbors=N_NEIGHBORS, n_jobs=-1),
        ),
    ]
)

logreg = Pipeline(
    steps=[
        ("prep", preprocess),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)

print("Pipelines ready.")

Pipelines ready.


## 4. Train & evaluate

In [15]:
def evaluate(name: str, model, X_te, y_te) -> dict:
    y_pred = model.predict(X_te)
    # Macro-averaged one-vs-rest ROC-AUC across all activities
    proba = model.predict_proba(X_te)
    classes = list(model.classes_)
    proba_ord = proba[:, [classes.index(c) for c in ACTIVITY_ORDER]]
    roc_auc_macro = float(
        roc_auc_score(
            y_te,
            proba_ord,
            multi_class="ovr",
            average="macro",
            labels=ACTIVITY_ORDER,
        )
    )
    metrics = {
        "model": name,
        "accuracy": float(accuracy_score(y_te, y_pred)),
        "f1_macro": float(f1_score(y_te, y_pred, average="macro", labels=ACTIVITY_ORDER, zero_division=0)),
        "f1_micro": float(f1_score(y_te, y_pred, average="micro", labels=ACTIVITY_ORDER, zero_division=0)),
        "f1_weighted": float(
            f1_score(y_te, y_pred, average="weighted", labels=ACTIVITY_ORDER, zero_division=0)
        ),
        "roc_auc_macro": roc_auc_macro,
    }
    print(f"\n=== {name} (test) ===")
    for k in ("accuracy", "f1_macro", "f1_micro", "f1_weighted", "roc_auc_macro"):
        print(f"{k:14s}: {metrics[k]:.4f}")
    print("\nClassification report:")
    print(
        classification_report(
            y_te, y_pred, labels=ACTIVITY_ORDER, digits=3, zero_division=0
        )
    )
    cm = confusion_matrix(y_te, y_pred, labels=ACTIVITY_ORDER)
    cm_df = pd.DataFrame(cm, index=ACTIVITY_ORDER, columns=ACTIVITY_ORDER)
    cm_path = ART / f"baseline_{name.lower().replace(' ', '_')}_confusion_matrix.csv"
    cm_df.to_csv(cm_path)
    print(f"Saved confusion matrix -> {cm_path}")
    return metrics, y_pred


print("Fitting kNN...")
knn.fit(X_train, y_train)
knn_metrics, knn_pred = evaluate("kNN", knn, X_test, y_test)

print("Fitting Logistic Regression...")
logreg.fit(X_train, y_train)
logreg_metrics, logreg_pred = evaluate("LogisticRegression", logreg, X_test, y_test)

summary = pd.DataFrame([knn_metrics, logreg_metrics]).set_index("model")
display(summary.round(4))
summary.to_csv(ART / "baseline_knn_logreg_metrics.csv")
print("Saved metrics ->", ART / "baseline_knn_logreg_metrics.csv")


Fitting kNN...

=== kNN (test) ===
accuracy      : 0.2902
f1_macro      : 0.2859
f1_micro      : 0.2902
f1_weighted   : 0.2869
roc_auc_macro : 0.6933

Classification report:
              precision    recall  f1-score   support

           A      0.360     0.379     0.369      1402
           B      0.619     0.706     0.660      1404
           C      0.393     0.457     0.422      1367
           D      0.227     0.331     0.270      1401
           E      0.263     0.330     0.293      1366
           F      0.243     0.297     0.267      1402
           G      0.192     0.196     0.194      1401
           H      0.151     0.135     0.143      1400
           I      0.152     0.149     0.150      1331
           J      0.204     0.155     0.177      1332
           K      0.196     0.158     0.175      1402
           L      0.078     0.075     0.076      1330
           M      0.356     0.327     0.341      1400
           O      0.319     0.260     0.287      1261
           P   

,accuracy,f1_macro,f1_micro,f1_weighted,roc_auc_macro
model,,,,,
kNN,0.2902,0.2859,0.2902,0.2869,0.6933
LogisticRegression,0.1845,0.1666,0.1845,0.1675,0.7521


Saved metrics -> /Users/tomhait/git/tom.aharon-hait/projects/phone_watch_project/artifacts/baseline_knn_logreg_metrics.csv


## 5. Optional: save test predictions

In [16]:
pred_df = test_win[["window_id", "user_id", "activity", "device", "sensor"]].copy()
pred_df = pred_df.rename(columns={"activity": "y_true"})
pred_df["y_pred_knn"] = knn_pred
pred_df["y_pred_logreg"] = logreg_pred
pred_path = ART / "baseline_test_predictions.csv"
pred_df.to_csv(pred_path, index=False)
print(f"Saved {pred_path} ({len(pred_df):,} rows)")
display(pred_df.head())

Saved /Users/tomhait/git/tom.aharon-hait/projects/phone_watch_project/artifacts/baseline_test_predictions.csv (24,772 rows)


,window_id,user_id,y_true,device,sensor,y_pred_knn,y_pred_logreg
0,1606_phone_accel_A_0000,1606,A,phone,accel,M,S
1,1606_phone_accel_A_0001,1606,A,phone,accel,C,S
2,1606_phone_accel_A_0002,1606,A,phone,accel,M,S
3,1606_phone_accel_A_0003,1606,A,phone,accel,C,S
4,1606_phone_accel_A_0004,1606,A,phone,accel,M,S


## 6. Results summary

Subject-held-out test set (10 users). Features per window: mean/std of `x`,`y`,`z` + one-hot `device`/`sensor`. Chance accuracy with 18 classes ≈ **5.6%**. Random ROC-AUC ≈ **0.50**.

| Model | Accuracy | F1 macro | F1 micro | F1 weighted | ROC-AUC macro |
|---|---:|---:|---:|---:|---:|
| **kNN (k=5)** | **0.290** | **0.286** | **0.290** | **0.287** | 0.693 |
| Logistic Regression | 0.184 | 0.167 | 0.184 | 0.168 | **0.752** |

### Metrics (short)
- **Accuracy:** share of windows with the correct activity label.
- **F1 (per class):** balances precision and recall for one activity; high when that class is found without many false alarms.
- **F1 macro:** average of the 18 per-class F1 scores — **each activity counts equally** (good for “do we recognize every class?”).
- **F1 micro:** one F1 from all predictions pooled; for single-label multiclass this equals **accuracy** (overall correctness).
- **F1 weighted:** like macro, but weighted by how many test windows each activity has (common classes matter more).
- **ROC-AUC macro:** for each activity, score one-vs-rest ROC-AUC from predicted probabilities, then **average equally across the 18 activities**. Measures ranking quality (can the model score the true class higher?), not only hard argmax accuracy. Chance ≈ 0.5.

### Takeaways
- **kNN wins on accuracy / F1** (~29% vs ~18%).
- **Logistic regression wins on ROC-AUC macro** (~0.75 vs ~0.69): better probability ranking even when the top predicted class is often wrong.
- Both models are **well above chance** on accuracy and AUC, so mean/std axis stats + device/sensor carry activity signal.
- Absolute hard-classification performance is still **modest**: 18 fine-grained classes are hard with only first-/second-order axis stats — no magnitude, PSD, or phone↔watch fusion.
- Natural next steps: add magnitude / spectral features, tune `k`, try LightGBM, or fuse phone+watch windows.
